In [41]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt




In [42]:
###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name. NOTE: You will
# likely need more variables for your constructor to handle the hostname and port of the MongoDB
# server, and the database and collection names
#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

shelter = AnimalShelter()

df = pd.DataFrame.from_records(shelter.read({}))

df.drop(columns=['_id'], inplace=True)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


In [43]:
#########################
# Dashboard Layout / View
#########################

app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),

    html.Center(
        html.B(
            html.H1('SNHU CS-340 Dashboard')
        )
    ),
    
    html.H3("Ahmed Ahmed - AAC Animal Shelter Dashboard"),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={
            'overflowX': 'auto'
        },
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '200px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        }
    ),

    html.Br(),

    html.Hr(),

    html.Div(
        id='map-id',
        className='col s12 m6'
    )
])

In [44]:
#############################################
# Interaction Between Components / Controller
#############################################

# This callback highlights the selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [
        {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        }
        for i in selected_columns
    ]

# This callback updates the geolocation map
# based on the row selected in the data table
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    dff = pd.DataFrame.from_dict(viewData)

    # If the table has no data, do not try to access a row
    if dff.empty:
        return [
            dl.Map(
                style={
                    'width': '1000px',
                    'height': '500px'
                },
                center=[30.75, -97.48],
                zoom=10,
                children=[
                    dl.TileLayer(id='base-layer-id')
                ]
            )
        ]

    # Select the first row if no row is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={
                'width': '1000px',
                'height': '500px'
            },
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),

                dl.Marker(
                    position=[
                        dff.iloc[row, 13],
                        dff.iloc[row, 14]
                    ],
                    children=[
                        dl.Tooltip(
                            dff.iloc[row, 4]
                        ),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(
                                dff.iloc[row, 9]
                            )
                        ])
                    ]
                )
            ]
        )
    ]

    return [
        dl.Map(
            style={
                'width': '1000px',
                'height': '500px'
            },
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),

                dl.Marker(
                    position=[
                        dff.iloc[row, 13],
                        dff.iloc[row, 14]
                    ],
                    children=[
                        dl.Tooltip(
                            dff.iloc[row, 4]
                        ),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(
                                dff.iloc[row, 9]
                            )
                        ])
                    ]
                )
            ]
        )
    ]

In [45]:
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://avatarpegasus-airlineaztec-3000.codio.io/proxy/8050/
